In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from IPython.display import Latex, HTML, Math, display
from uncertainties import ufloat
from uncertainties.umath import sqrt
from uncertainties import unumpy as unp
from scipy.stats import linregress
from scipy.optimize import curve_fit
from uncertainties.umath import sin, radians 
from uncertainties.umath import *

In [ ]:
# 1. Brewster Winkel

#senkrechte Polarisation
I_senk = unp.uarray([], )           #[[mA] gemessene Intensität bei senkrechtem Lichteinfall
alpha_senk = np.array([35, 40, 45, 50, 55, 60, 65])         #[°] Einfallswinkel


#plot senkrecht
I_senk_nom  = unp.nominal_values(I_senk)
I_senk_u = unp.std_devs(I_senk)

plt.figure()
plt.errorbar(alpha_senk, I_senk_nom, yerr=I_senk_u, fmt='o', capsize=3, label="Data")
plt.xlabel(r"$Einfallswinkel \alpha [°]$")
plt.ylabel(r"Intensität I [mA]")
plt.legend()
plt.grid(True)
plt.show()

#parallele Polarisation
I_par = unp.uarray([], )            #[mA] Intensität bei paralleler Polarisation
alpha_par = np.array([23, 40, 45, 50, 55, 60, 65])          #[°] Einfalsswinkel; bei Mimimum noch Messwerte hinzufügen


#plot parallel
I_par_nom = unp.nominal_values(I_par)
I_par_u = unp.std_devs(I_par)

coeffs = np.polyfit(alpha_par, I_par, 3)                #evtl kubisch fitten? je nachdem wie die werte ausschauen
poly = np.poly1d(coeffs)

alpha_fit = np.linspace(np.min(alpha_par), np.max(alpha_par), 300)
I_fit = poly(alpha_fit)


plt.figure()
plt.errorbar(alpha_par, I_par_nom, yerr=I_par_u, fmt='o', capsize=3, label="Data")
plt.plot(alpha_fit, I_fit, linestyle='--', label="cubic fit")
plt.xlabel(r"$Einfallswinkel \alpha [°]$")
plt.ylabel(r"Intensität I [mA]")
plt.legend()
plt.grid(True)
plt.show()


#berechnung brewster winkel
I_min = np.argmin(I_par_nom)        #index von kleinsten intensitärswert
alpha_B = alpha_par[I_min]          #brewster winkel: dort wo die Intensität minimal ist
alpha_B_u = 2                       #abgeschätze Unsicherheit vom brewster winkel 

alpha_B_rad = np.deg2rad(alpha_B)       #[°] -->[rad]
alpha_B_u_rad = np.deg2rad(alpha_B_u)   #[°] -->[rad]

n1 =                                #Brechungsindex Luft (oder Vakuum verwenden? --> fragen)

n2 = n1 * np.tan(alpha_B_rad)       #Brechungsindexplatte
n2_u = (1 / (np.cos(alpha_B_rad)**2)) * alpha_B_u_rad           #Unsicherheit Brechungsindex platte (Ka wieso - abegschrieben von aleks)


#ergebnisse printen
Brewster = ufloat(alpha_B, alpha_B_u)
Brechungsindex = ufloat(n2, n2_u)

print(f"Brewsterwinkel: {Brewster}")
print(f"Brechungsindex Platte: {Brechungsindex}")




In [ ]:
# 2. Spannungsoptik

#Werte
lam = *10**(-6)             #[mm] Wellenlänge verwendetes Licht

A_kolben =                  #kolbenfläche Hydraulikpresse [mm²]

l1 = ufloat(, )             #Seitenlänge 1 der Probe [mm]
l2 = ufloat(, )             #Seitenlänge 2 der Probe [mm]
A_probe = l1 * l2           #Oberfläche der Probe [mm²]

d = ufloat(, )              #Dicke der probe am Weg den das Licht nimmt [mm]

p_kolben = unp.uarray([], )         #Druck auf den Kolben der Presse bei verschiedenen Ordnungen (0, 1, 2) [Pa]


#sigma, C und S berechnen
sigma = (p_kolben * A_kolben) / A_probe     #Druck auf die Probe bei verschiedenen Ordnungen [Pa] 

C = lam / (sigma * d)                       #[1/Pa] Materialkonstante
C_mean = np.mean(unp.nominal_values(C))

S = lam / C_mean                            #[N/mm] Spannungsoptische Konstante


#werte ausgeben
df = pd.DataFrame({
    "p_kolben [Pa]": p_kolben,
    "sigma_probe [Pa]": sigma,
    "C": C
})

print()
print(f"C Mittel: {C_mean}")
print(f"S: {S}")

In [ ]:
# 3. Drehung der Polarisationsebene (Optische Aktivität)

#beim quarz nur schauen ob im/gegen Uhrezeigersinn (= rechts/links drehend)

#Drehwinkel Zuckerlösung
mw = ufloat(, )                 #[g = ml] Masse Wasser; sollen ca 20g sein
ms = ufloat(, )                 #[g] Masse Zucker; sollen ca. 2g sein
C = ms / ms                     #Verhältnis = Konzentration; einheitslos
l = (2, 0.2)                    #[dm] Länge der Küvette, gegeben (MUSS dm sein  - Konvention)

a0 = ufloat(, 0.05)             #[°]; Winkel bei neutraler Position
ae = ufloat(, 0.05)             #[°]; Winkel nach Messung (erneut dunkles Bild)
a = ae - a0                     #[°]; Winkel um den verdreht wurde = Drehwinkel; WICHTIG! WENIGER als 20°

_a_ = a / (C * l)               #spezifischer Drehwinkel; [°/dm]

#Ergebnisse ausgeben
print(f"spezifischer Drehwinkel Zuckerlösung: {_a_}")

#Außerdem DRehrichtung notieren: ENTGEGENGESETZT der Richtung in die sich der Nonius gedreht hat!!

